In [7]:
from google.cloud import bigquery
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

In [8]:
client = bigquery.Client(project="pacey32-agency")

sql = """
WITH profile AS (
    SELECT
        playerId,
        ANY_VALUE(player_name) AS player,
        ANY_VALUE(age) AS age,
        ANY_VALUE(position) AS position,
        ANY_VALUE(heightInCentimeters) AS height_cm,
        ANY_VALUE(weightInKilograms) AS weight_kg,
        ANY_VALUE(draftRound) AS draftRound,
        ANY_VALUE(draftOverall) AS draftPick,
        ANY_VALUE(draftYear) AS draftYear,
        ANY_VALUE(draftTeam) AS draftTeam,
        ANY_VALUE(shoots_catches) AS shoots_catches,
        ANY_VALUE(birth_country) AS birthCountry
    FROM `pacey32-agency.Comparison.01_PlayerProfile`
    WHERE activeFlag = 1
    GROUP BY playerId
),

stats AS (
    SELECT
        playerId,
        season,
        SUM(games_played) AS games,
        SUM(goals) AS goals,
        SUM(assists) AS assists,
        SUM(points) AS points,
        SAFE_DIVIDE(SUM(goals), SUM(games_played)) AS goals_per_game,
        SAFE_DIVIDE(SUM(assists), SUM(games_played)) AS assists_per_game,
        SAFE_DIVIDE(SUM(points), SUM(games_played)) AS points_per_game,
        SAFE_DIVIDE(SUM(toi_minutes), SUM(games_played)) AS avg_toi_minutes,
        SAFE_DIVIDE(SUM(goals_per_60 * toi_minutes), SUM(toi_minutes)) AS goals_per_60,
        SAFE_DIVIDE(SUM(assists_per_60 * toi_minutes), SUM(toi_minutes)) AS assists_per_60,
        SAFE_DIVIDE(SUM(points_per_60 * toi_minutes), SUM(toi_minutes)) AS points_per_60
    FROM `pacey32-agency.Comparison.03_PlayerSeasonStats`
    WHERE seasonPart = 'RegularSeason'
    GROUP BY playerId, season
),

latest AS (
    SELECT *
    FROM stats
    WHERE games >= 40
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY playerId
        ORDER BY season DESC
    ) = 1
)

SELECT
    p.*,
    s.season,
    s.games,
    s.goals,
    s.assists,
    s.points,
    s.goals_per_game,
    s.assists_per_game,
    s.points_per_game,
    s.avg_toi_minutes,
    s.goals_per_60,
    s.assists_per_60,
    s.points_per_60
FROM profile p
LEFT JOIN latest s
    ON p.playerId = CAST(s.playerId AS INT64)
"""

df_players = client.query(sql).to_dataframe()

In [9]:
features = [
    "age",
    "height_cm",
    "weight_kg",
    "draftPick",
    "goals_per_game",
    "assists_per_game",
    "points_per_game",
    "avg_toi_minutes",
    "goals_per_60",
    "assists_per_60",
    "points_per_60"
]

df_model = df_players.dropna(subset=features).copy()

print(f"Players available for comparison: {len(df_model):,}")

Players available for comparison: 659


In [10]:
scaler = StandardScaler()

X = scaler.fit_transform(df_model[features])

similarity_matrix_v2 = cosine_similarity(X)

In [12]:
def find_comparables(player_id, n=10):
    idx = df_model.index[df_model["playerId"] == player_id][0]
    pos = df_model.index.get_loc(idx)

    target_position = df_model.loc[idx, "position"]
    target_hand = df_model.loc[idx, "shoots_catches"]

    results = df_model[[
        "playerId", "player", "age", "position", "shoots_catches",
        "draftPick", "games", "goals", "assists", "points",
        "goals_per_game", "assists_per_game", "points_per_game",
        "avg_toi_minutes", "goals_per_60", "assists_per_60", "points_per_60",
        "height_cm", "weight_kg"
    ]].copy()

    results["base_similarity"] = similarity_matrix_v2[pos]
    results["same_hand"] = (results["shoots_catches"] == target_hand).astype(int)

    hand_weight = 0.05 if target_position == "D" else 0.02

    results["similarity"] = (
        results["base_similarity"] * (1 - hand_weight) +
        results["same_hand"] * hand_weight
    )

    results = results[
        (results["playerId"] != player_id) &
        (results["position"] == target_position)
    ]

    results = results.sort_values("similarity", ascending=False).head(n)

    results["base_similarity"] = (results["base_similarity"] * 100).round(1)
    results["similarity"] = (results["similarity"] * 100).round(1)

    target = df_model.loc[[idx], results.columns.intersection(df_model.columns)].copy()

    target["base_similarity"] = 100.0
    target["same_hand"] = 1
    target["similarity"] = 100.0

    results = pd.concat([target, results], ignore_index=True)

    return results

In [14]:
display(find_comparables(8480069, 10))

,playerId,player,age,position,shoots_catches,draftPick,games,goals,assists,points,...,points_per_game,avg_toi_minutes,goals_per_60,assists_per_60,points_per_60,height_cm,weight_kg,base_similarity,same_hand,similarity
0,8480069,Cale Makar,28,D,R,4,75,20,59,79,...,1.053333,24.848000,0.640000,1.900000,2.540000,183,85,100.0,1,100.0
1,8480803,Evan Bouchard,27,D,R,10,82,21,75,96,...,1.170732,24.678049,0.620000,2.220000,2.850000,191,87,94.3,1,94.6
2,8479323,Adam Fox,28,D,R,66,55,9,46,55,...,1.000000,23.629091,0.420000,2.120000,2.540000,180,84,92.8,1,93.2
3,8474578,Erik Karlsson,36,D,R,15,75,18,56,74,...,0.986667,23.600000,0.610000,1.900000,2.510000,183,84,89.9,1,90.4
4,8479325,Charlie McAvoy,29,D,R,14,69,11,53,64,...,0.927536,24.389855,0.390000,1.890000,2.280000,185,96,89.1,1,89.6
5,8480800,Quinn Hughes,27,D,L,7,74,7,69,76,...,1.027027,27.737838,0.202629,2.017983,2.220612,178,82,91.9,0,87.3
6,8478460,Zach Werenski,29,D,L,8,75,24,60,84,...,1.120000,26.614667,0.720000,1.800000,2.520000,188,97,89.4,0,84.9
7,8482105,Jake Sanderson,24,D,L,5,67,16,43,59,...,0.880597,24.838806,0.580000,1.550000,2.130000,188,92,89.1,0,84.7
8,8480839,Rasmus Dahlin,26,D,L,1,77,19,55,74,...,0.961039,24.185714,0.610000,1.770000,2.380000,191,93,89.0,0,84.5
9,8480036,Miro Heiskanen,27,D,L,3,77,9,55,64,...,0.831169,25.472727,0.280000,1.680000,1.960000,188,89,88.5,0,84.0
